# Duration Core MVP

Provisional historical inspection of the accepted Duration Core definition. These seven-level scores are ordinal economic judgments, not return estimates or ETF rankings.

Run the calculation section to explicitly acquire, calculate, and save all three exposure histories. After restarting the kernel, the visualization section can run on its own using only the saved CSV and JSON. See `README.md` for commands and cache behavior.

## Calculation — explicit acquisition through persistence

In [ ]:
from pathlib import Path
from duration_config import load_config
from fred_data import acquire_fred
from duration_components import calculate_components
from duration_core import evaluate_core
from duration_results import save_results

BASE = Path.cwd()
config = load_config(BASE / "duration_config.yaml")
REFRESH_FRED = False  # Set True for an explicit fresh download; False reuses valid caches.

In [ ]:
# 1. Acquire Raw Observations; the local raw cache is separate from Core results.
observations, source_snapshot = acquire_fred(BASE / "cache", refresh=REFRESH_FRED)
print(f"Loaded {len(observations):,} source observations across all three Treasury tenors.")

In [ ]:
# 2. Calculate tenor-local Features, Component Values, and Component States.
components = calculate_components(observations, config["trend"], config["recent_move"])
print("Components calculated for all tenors; components retains M5 and endpoint lineage.")

In [ ]:
# 3. Map already calculated Components to all three synthetic Duration Exposures.
core_results = evaluate_core(components, config["exposures"], config["rules"])
print(f"Completed {core_results['core_score'].notna().sum():,} Core Duration judgments.")

In [ ]:
# 4. Persist completed results and provenance before using visualization controls.
saved_csv, saved_metadata = save_results(core_results, source_snapshot, config, BASE / "output")
print(f"Saved {saved_csv.name} and {saved_metadata.name}.")

## Visualization — runnable alone after a kernel restart

Run every cell below without executing the calculation section. This loads the saved snapshot; subsequent YAML edits do not change it. The saved metadata supplies the historical-event presets. To require the current analytical configuration to match, explicitly call `load_results(..., expected_config=load_config(...))`; a mismatch raises and requires a new calculation/save.

The default Custom period uses shared available dates. You can extend the dates to inspect an exposure's earlier history. Event presets preserve previous Custom dates. Gaps and warmup observations are unavailable, and no lines bridge structural discontinuities.

In [ ]:
from pathlib import Path
from IPython.display import display
from duration_results import load_results
from duration_plot import DurationControls

# 5. Load persisted analytical results independently of acquisition/calculation.
loaded_results, saved_metadata = load_results(Path.cwd() / "output")
print(f"Saved snapshot generated at {saved_metadata['generated_at_utc']}.")
print(f"Calculation fingerprint: {saved_metadata['calculation_fingerprint']}")

In [ ]:
# 6. Controls only filter and display the loaded rows.
controls = DurationControls(loaded_results, saved_metadata["configuration"]["historical_events"])
display(controls.ui)
controls.render()